# Labrador Sea SE-region seasonal timeseries — CESM3 preindustrial development

For each of the six diagnostic fields, plot the seasonal (JFM, JAS, and if you like all four standard seasons) area-weighted mean over the SE Labrador Sea box defined in `labsea_utils.REGIONS['LabSea_SE']`, for the **entire simulated period** of each run.

Runs are shown together on the same axis:
* **388** — Lab Sea stays frozen
* **377** — Lab Sea melts

Both individual seasonal values and a smoothed line are drawn.

In [11]:
%load_ext autoreload
%autoreload 2

import importlib
import labsea_utils as lsu
importlib.reload(lsu)

import matplotlib.pyplot as plt
from dask.distributed import Client
from dask_jobqueue import PBSCluster

RUNS    = ["388", "377"]
VARS    = lsu.VAR_LIST                # [ICEFRAC, SHFLX, LHFLX, TS, PRECT, CLDLOW]
SEASONS = ["JFM", "AMJ", "JAS", "OND"]
REGION  = "LabSea_SE"
FIG_DIR = "./figs"
ROLL    = 5    # years, centered rolling mean for the highlighted line

for r in RUNS:
    y0, y1 = lsu.year_range(r)
    print(f"Run {r}: years {y0:04d}-{y1:04d}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Run 388: years 0001-0056
Run 377: years 0001-0151


In [12]:
cluster = PBSCluster(
    account="P03010039",
    interface="ext",
    walltime="12:00:00",
    queue="main",   
    cores=4,
    memory="32GB",
    processes=4,      # one process per core (safe default)cesm2
    local_directory="/glade/derecho/scratch/rneale/dask-temp",
    log_directory="/glade/derecho/scratch/rneale/dask-logs"
)

cluster.scale(jobs=32)
client = Client(cluster)

client

/glade/work/rneale/conda-envs/neale_vproc2/lib/python3.14/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 44165 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: /node/crhtc44.hpc.ucar.edu/6809/proxy/44165/status,
Dashboard: /node/crhtc44.hpc.ucar.edu/6809/proxy/44165/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.94:42895,Workers: 0
Dashboard: /node/crhtc44.hpc.ucar.edu/6809/proxy/44165/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [ ]:
# Each run uses its own full period.
datasets = {}
for r in RUNS:
    print(f"Opening run {r} full record ...")
    datasets[r] = lsu.load_run(r, VARS)
    print(f"  {r}: {datasets[r].sizes['time']} monthly records")

Opening run 388 full record ...
  388: 672 monthly records
Opening run 377 full record ...


In [ ]:
# Compute area-weighted monthly regional-mean timeseries once per (var, run).
monthly_ts = {var: {} for var in VARS}
for var in VARS:
    for r in RUNS:
        ts = lsu.regional_mean_ts(datasets[r], var, REGION)
        ts.load()
        monthly_ts[var][r] = ts
    print(f"  built monthly regional-mean timeseries for {var}")

# GPCP monthly regional-mean for PRECT — full obs record, aligned by index
# onto the model year axis when plotted.
gpcp = lsu.load_gpcp()
monthly_ts["PRECT"]["GPCP"] = lsu.obs_regional_mean_ts(gpcp, "PRECT", REGION).load()
print(f"  added GPCP monthly ts (n={monthly_ts['PRECT']['GPCP'].sizes['time']})")

In [ ]:
# One figure per (var, season): seasonal-mean line + smoothed line for each run.
# Any obs entry (e.g. GPCP for PRECT) is drawn along the model x-axis starting
# at the first model year — the actual obs years are disregarded.
for var in VARS:
    for season in SEASONS:
        ts_by_run = {}
        for case_id in list(monthly_ts[var].keys()):   # RUNS + any obs (GPCP)
            ts_by_run[case_id] = lsu.to_seasonal_ts(monthly_ts[var][case_id], season)
        lsu.plot_seasonal_ts(ts_by_run, var, season, REGION,
                             save_dir=FIG_DIR, roll=ROLL)
        plt.show()